In [ ]:
import time
import numpy as np
from joblib import Parallel, delayed

def fwht(x):
    x = np.asarray(x, dtype=float).copy()
    L = len(x)
    if L == 0 or (L & (L - 1)) != 0:
        raise ValueError("Length of x must be a power of 2.")
    h = 1
    while h < L:
        y = x.reshape(-1, 2 * h)
        a = y[:, :h].copy()
        b = y[:, h:2*h].copy()
        y[:, :h] = a + b
        y[:, h:2*h] = a - b
        h *= 2
    return x

def gray_permutation(L):
    i = np.arange(L, dtype=np.int64)
    return i ^ (i >> 1)

def find_alphas_fast(target):
    target = np.asarray(target, dtype=complex)
    d = len(target)
    n = int(np.round(np.log2(d)))
    prob_level = np.abs(target)**2
    phase_level = np.angle(target)
    global_phase = 2.0 * np.mean(phase_level)
    alphas_y, alphas_z = [], []

    for j in range(n):
        left_prob, right_prob = prob_level[0::2], prob_level[1::2]
        parent_prob = left_prob + right_prob
        ratio_sq = np.zeros_like(parent_prob)
        nonzero = parent_prob > 0.0
        ratio_sq[nonzero] = right_prob[nonzero] / parent_prob[nonzero]
        ratio_sq = np.clip(ratio_sq, 0.0, 1.0)
        alphas_y.append(2.0 * np.arcsin(np.sqrt(ratio_sq)))

        left_phase, right_phase = phase_level[0::2], phase_level[1::2]
        alphas_z.append((right_phase - left_phase) / (2**j))

        prob_level = parent_prob
        phase_level = left_phase + right_phase

    return alphas_y, alphas_z, global_phase

def thetas_from_alphas_fast(alphas):
    thetas = []
    for alpha in alphas:
        alpha = np.asarray(alpha, dtype=float)
        L = len(alpha)
        if L == 1:
            thetas.append(alpha.copy())
            continue
        transformed = fwht(alpha)
        gray = gray_permutation(L)
        thetas.append(transformed[gray] / L)
    return thetas

def thetas_from_alphas_direct(alphas):
    thetas = []
    for alpha in alphas:
        alpha = np.asarray(alpha, dtype=float)
        L = len(alpha)
        if L == 1:
            thetas.append(alpha.copy())
            continue
        i = np.arange(L, dtype=np.uint64)
        j = np.arange(L, dtype=np.uint64)
        gray_i = i ^ (i >> np.uint64(1))
        values = gray_i[:, None] & j[None, :]
        parity = values.copy()
        for shift in [32, 16, 8, 4, 2, 1]:
            parity ^= parity >> np.uint64(shift)
        parity &= np.uint64(1)
        M = 1.0 - 2.0 * parity.astype(float)
        thetas.append((M @ alpha) / L)
    return thetas

def run_single_trial(m, compute_direct):
    """Runs a single trial for degree d = 2^m."""
    d = 2**m
    target = np.random.randn(d) + 1j * np.random.randn(d)
    target /= np.linalg.norm(target)

    # Fast Method Timing
    start_fast = time.perf_counter()
    alphas_y, alphas_z, _ = find_alphas_fast(target)
    fast_y = thetas_from_alphas_fast(alphas_y)
    fast_z = thetas_from_alphas_fast(alphas_z)
    fast_time = time.perf_counter() - start_fast

    direct_time = np.nan
    max_err = np.nan

    # Direct Method Timing & Error Calculation
    if compute_direct:
        start_direct = time.perf_counter()
        direct_y = thetas_from_alphas_direct(alphas_y)
        direct_z = thetas_from_alphas_direct(alphas_z)
        direct_time = time.perf_counter() - start_direct

        err_y = max(np.max(np.abs(f - d_)) for f, d_ in zip(fast_y, direct_y))
        err_z = max(np.max(np.abs(f - d_)) for f, d_ in zip(fast_z, direct_z))
        max_err = max(err_y, err_z)

    return fast_time, direct_time, max_err

if __name__ == '__main__':
    ms = list(range(2, 27)) # m = 2 to 18 (d up to 262,144)
    num_trials = 10        # Independent simulations per degree
    max_direct_m = 12       # Only run direct method up to m=12 (O(d^2) gets too slow)

    all_fast_times = []
    all_direct_times = []
    all_errors = []

    print(f"Starting simulations using joblib... ({num_trials} trials per degree)")

    for m in ms:
        compute_direct = (m <= max_direct_m)
        print(f"Processing m={m:02d} (d={2**m:6d}) | Direct method: {compute_direct}")

        # Parallelize the trials for the current m across all CPU cores (n_jobs=-1)
        results = Parallel(n_jobs=-1)(
            delayed(run_single_trial)(m, compute_direct) for _ in range(num_trials)
        )

        # Unpack results
        fast_times, direct_times, errors = zip(*results)

        all_fast_times.append(fast_times)
        all_direct_times.append(direct_times)
        all_errors.append(errors)

    # Save data to disk for plotting later
    np.savez('simulation_data.npz',
             ms=ms,
             fast_times=all_fast_times,
             direct_times=all_direct_times,
             errors=all_errors)

    print("\n[SUCCESS] Data generated and saved to 'simulation_data.npz'")